# Lab 08 solution, notebook 1: two tools in a chain, and a third step in the workflow

The worked answer to the challenge of `08a_meet_google_adk.ipynb`. The setup repeats what the notebook built
(`llm`, `reply_text`, `ask` and `get_weather`), so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import sys

# The solution sits in solved/; the agent files are one folder up, in my_agent/
sys.path.insert(0, str(Path.cwd().parent))

In [3]:
from google.adk.agents import Agent
from google.adk.integrations.openai import OpenAIResponsesLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

llm = OpenAIResponsesLlm(model=MODEL)


def reply_text(event):
    """The text of an event: its text parts joined, leaving out the model's thoughts."""
    return "".join(part.text for part in event.content.parts if part.text and not part.thought)


async def ask(runner, session, text):
    """Send one user message in a session and return the events of the run."""
    message = types.Content(role="user", parts=[types.Part(text=text)])
    events = []
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id, new_message=message):
        events.append(event)
    return events

In [4]:
import requests


def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo."""
    print(f"🌤️ TOOL CALLED: get_weather({city})")

    # 1. Find the city's coordinates
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    )
    locations = geo.json().get("results", [])
    if not locations:
        return f"City not found: {city}"
    location = locations[0]

    # 2. Get the current weather at those coordinates
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current": "temperature_2m,apparent_temperature,precipitation,wind_speed_10m",
        },
        timeout=10,
    )

    return str({"city": location["name"], "country": location["country"], **weather.json()["current"]})

## Task 1: chain two tools

`to_fahrenheit` is one line of arithmetic with a `TOOL CALLED` print, so that we see when it runs. The agent gets
both functions in `tools`, and one more line of instruction that says when to use the new one.

In [5]:
def to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from degrees Celsius to degrees Fahrenheit."""
    print(f"🌡️ TOOL CALLED: to_fahrenheit({celsius})")
    return round(celsius * 9 / 5 + 32, 1)


fahrenheit_agent = Agent(
    name="weather_agent",
    model=llm,
    instruction=(
        "You are a friendly weather reporter. "
        "Always use get_weather for current conditions. "
        "Always use to_fahrenheit to convert a temperature; never convert it yourself. "
        "Keep your reports short and fun."
    ),
    tools=[get_weather, to_fahrenheit],
)

fahrenheit_runner = InMemoryRunner(agent=fahrenheit_agent, app_name="playground")
fahrenheit_session = await fahrenheit_runner.session_service.create_session(app_name="playground", user_id="student")

events = await ask(fahrenheit_runner, fahrenheit_session, "What's the temperature in Lisbon right now, in Fahrenheit?")

for number, event in enumerate(events, start=1):
    for call in event.get_function_calls():
        print(number, "function call    :", call.name, call.args)
    for response in event.get_function_responses():
        print(number, "function response:", response.name, response.response)
    if event.is_final_response():
        print(number, "final answer     :", reply_text(event))

🌤️ TOOL CALLED: get_weather(Lisbon)


🌡️ TOOL CALLED: to_fahrenheit(19.4)


1 function call    : get_weather {'city': 'Lisbon'}
2 function response: get_weather {'result': "{'city': 'Lisbon', 'country': 'Portugal', 'time': '2026-11-13T14:15', 'interval': 900, 'temperature_2m': 19.4, 'apparent_temperature': 18.6, 'precipitation': 0.0, 'wind_speed_10m': 15.3}"}
3 function call    : to_fahrenheit {'celsius': 19.4}
4 function response: to_fahrenheit {'result': 66.9}
5 final answer     : It’s about **67°F** in Lisbon right now—pleasantly mild!


Five events, and `TOOL CALLED` printed twice:

1. The model asked for `get_weather` for Lisbon.
2. The response held the temperature, 19.4.
3. The model asked for `to_fahrenheit` with `{'celsius': 19.4}`: the number from event 2.
4. The response was 66.9. The function returns a number, not a dictionary, so ADK put it under the key `result`.
5. The answer gives 67 °F.

**Notice** that the two calls came in two separate events. The second call could not be written together with the
first, because its argument is a value from the first call's response. We wrote no code that passes the
temperature from one function to the other: the model read the response and wrote the next call's argument from
it.

## Task 2: add a third step to the workflow

`pack_agent` gets one sentence about an afternoon in a city and lists three things to pack. The new workflow's
edges go from `START` through `pick_city` and `plan_agent` to it.

In [6]:
from google.adk.workflow import START, Workflow
from my_agent.surprise_trip.agent import pick_city, plan_agent

pack_agent = Agent(
    name="pack_agent",
    model=llm,
    instruction="You get one sentence about an afternoon in a city. List three things to pack for it, on one line.",
)

packed_trip = Workflow(
    name="packed_trip",
    edges=[(START, pick_city, plan_agent, pack_agent)],
)

packed_runner = InMemoryRunner(agent=packed_trip, app_name="playground")
packed_session = await packed_runner.session_service.create_session(app_name="playground", user_id="student")

events = await ask(packed_runner, packed_session, "go")

for event in events:
    text = event.output if event.content is None else reply_text(event)
    print(f"{event.author:13} | {text}")

packed_trip   | Cape Town
plan_agent    | Cape Town is a beautiful place to spend a free afternoon strolling through the Kirstenbosch Botanical Garden.
pack_agent    | Comfortable walking shoes, a water bottle, sunscreen.


Three events, in the order of the edges: the function returned *Cape Town*, `plan_agent` suggested an afternoon in
Kirstenbosch National Botanical Garden, and `pack_agent` listed walking shoes, sunscreen and a water bottle.

**Notice** how the city reached the third step. In a `Workflow` each step receives only what the step before it
returned. `pack_agent` never received the function's output. It received `plan_agent`'s sentence, which names the
city. That is why the instruction of `plan_agent` says *Name the city*.